In [87]:
pip install pandas numpy


Note: you may need to restart the kernel to use updated packages.


In [88]:
import os
import random
import pandas as pd
from pandas.errors import EmptyDataError


In [89]:
products_path = "data/products.csv"
feedback_path = "storage/feedback.csv"
expected_cols = ["user_id", "product_id", "shown", "clicked"]


products_df = pd.read_csv(products_path)
products_df.head()

,product_id,product_name,category,brand,price,rating,description
0,1,Wireless Headphones,Electronics,SonicX,79.99,4.4,Noise cancelling wireless headphones with long...
1,2,Bluetooth Speaker,Electronics,SoundBeat,49.99,4.2,Portable speaker with deep bass and waterproof...
2,3,Smart Watch,Electronics,TimeSync,129.99,4.3,Fitness tracking smart watch with heart rate m...
3,4,Running Shoes,Fashion,RunFast,89.99,4.5,Lightweight running shoes for daily training
4,5,Hoodie,Fashion,UrbanFit,39.99,4.1,Comfortable cotton hoodie for casual wear


In [90]:
def ensure_feedback_file(path=feedback_path):
    recreate = False

    if not os.path.exists(path) or os.path.getsize(path) == 0:
        recreate = True
    else:
        try:
            test_df = pd.read_csv(path)
            if test_df.columns.tolist() != expected_cols:
                recreate = True
        except (EmptyDataError, Exception):
            recreate = True

    if recreate:
        pd.DataFrame(columns=expected_cols).to_csv(path, index=False)

    return pd.read_csv(path)


feedback_df = ensure_feedback_file()
print(feedback_df.columns.tolist())
feedback_df.head()


['user_id', 'product_id', 'shown', 'clicked']


,user_id,product_id,shown,clicked


In [91]:
EPSILON = 0.2  # 20% explore, 80% exploit


def load_feedback(path=feedback_path):
    return pd.read_csv(path)


def get_products_by_category(products, category):
    if not category or category == "All":
        return products.copy()
    return products[products["category"] == category].copy()


def get_user_preference(user_id, feedback, products):
    if feedback.empty:
        return None

    clicked = feedback[
        (feedback["user_id"] == user_id) &
        (feedback["clicked"] == 1)
    ]

    if clicked.empty:
        return None

    merged = clicked.merge(
        products[["product_id", "category"]],
        on="product_id",
        how="left"
    )

    if merged.empty:
        return None

    return merged["category"].mode().iloc[0]


In [92]:
"""Product statistics
- how many times a product was shown
- how many times it was clicked
- click rate = clicks / impressions"""

def get_product_stats(products, feedback):
    stats = products[["product_id", "product_name", "category", "brand", "price", "rating"]].copy()

    if feedback.empty:
        stats["shown_count"] = 0
        stats["click_count"] = 0
        stats["click_rate"] = 0.0
        return stats

    grouped = feedback.groupby("product_id")[["shown", "clicked"]].sum().reset_index()
    grouped = grouped.rename(columns={"shown": "shown_count", "clicked": "click_count"})

    stats = stats.merge(grouped, on="product_id", how="left").fillna(0)
    stats["shown_count"] = stats["shown_count"].astype(int)
    stats["click_count"] = stats["click_count"].astype(int)

    stats["click_rate"] = stats.apply(
        lambda row: row["click_count"] / row["shown_count"] if row["shown_count"] > 0 else 0.0,
        axis=1
    )

    return stats


In [93]:
stats_df = get_product_stats(products_df, feedback_df)
stats_df.head()


,product_id,product_name,category,brand,price,rating,shown_count,click_count,click_rate
0,1,Wireless Headphones,Electronics,SonicX,79.99,4.4,0,0,0.0
1,2,Bluetooth Speaker,Electronics,SoundBeat,49.99,4.2,0,0,0.0
2,3,Smart Watch,Electronics,TimeSync,129.99,4.3,0,0,0.0
3,4,Running Shoes,Fashion,RunFast,89.99,4.5,0,0,0.0
4,5,Hoodie,Fashion,UrbanFit,39.99,4.1,0,0,0.0


In [94]:
"""Epsilon-greedy bandit logic

- with probability ε → explore (pick a random product)
- otherwise → exploit (pick the best known product)"""

def choose_recommendations(user_id, products, feedback, category="All", top_n=3, epsilon=EPSILON):
    user_preference = get_user_preference(user_id, feedback, products)

    if category != "All":
        candidates = get_products_by_category(products, category)
    elif user_preference:
        candidates = get_products_by_category(products, user_preference)
    else:
        candidates = products.copy()

    stats = get_product_stats(products, feedback)[["product_id", "shown_count", "click_count", "click_rate"]]
    candidates = candidates.merge(stats, on="product_id", how="left").fillna(0)

    if candidates.empty:
        return []

    recommendations = []
    available = candidates.copy()

    for _ in range(min(top_n, len(available))):
        if random.random() < epsilon:
            selected = available.sample(1).iloc[0]   # explore
        else:
            selected = available.sort_values(
                by=["click_rate", "rating", "shown_count"],
                ascending=[False, False, True]
            ).iloc[0]   # exploit

        recommendations.append(selected.to_dict())
        available = available[available["product_id"] != selected["product_id"]]

    return recommendations


In [95]:
"""Feedback logging
- impressions when a product is shown
- clicks when a user interacts with a product"""

def save_feedback(user_id, product_id, shown, clicked, path=feedback_path):
    row = pd.DataFrame([{
        "user_id": user_id,
        "product_id": product_id,
        "shown": shown,
        "clicked": clicked
    }])
    row.to_csv(path, mode="a", index=False, header=False,lineterminator="\n")


def log_impression(user_id, product_id, path=feedback_path):
    save_feedback(user_id, product_id, shown=1, clicked=0, path=path)


def log_click(user_id, product_id, path=feedback_path):
    save_feedback(user_id, product_id, shown=0, clicked=1, path=path)


In [96]:
user_id = "user_001"
category = "Electronics"
top_n = 3

feedback_df = load_feedback()
recommendations = choose_recommendations(
    user_id=user_id,
    products=products_df,
    feedback=feedback_df,
    category=category,
    top_n=top_n
)

recommendations


[{'product_id': 1,
  'product_name': 'Wireless Headphones',
  'category': 'Electronics',
  'brand': 'SonicX',
  'price': 79.99,
  'rating': 4.4,
  'description': 'Noise cancelling wireless headphones with long battery life',
  'shown_count': 0,
  'click_count': 0,
  'click_rate': 0.0},
 {'product_id': 3,
  'product_name': 'Smart Watch',
  'category': 'Electronics',
  'brand': 'TimeSync',
  'price': 129.99,
  'rating': 4.3,
  'description': 'Fitness tracking smart watch with heart rate monitor',
  'shown_count': 0,
  'click_count': 0,
  'click_rate': 0.0},
 {'product_id': 2,
  'product_name': 'Bluetooth Speaker',
  'category': 'Electronics',
  'brand': 'SoundBeat',
  'price': 49.99,
  'rating': 4.2,
  'description': 'Portable speaker with deep bass and waterproof design',
  'shown_count': 0,
  'click_count': 0,
  'click_rate': 0.0}]

In [97]:
for item in recommendations:
    log_impression(user_id, int(item["product_id"]))

feedback_df = load_feedback()
feedback_df.tail()


,user_id,product_id,shown,clicked
0,user_001,1,1,0
1,user_001,3,1,0
2,user_001,2,1,0


In [98]:
if recommendations:
    clicked_product_id = int(recommendations[0]["product_id"])
    log_click(user_id, clicked_product_id)

feedback_df = load_feedback()
feedback_df.tail()


,user_id,product_id,shown,clicked
0,user_001,1,1,0
1,user_001,3,1,0
2,user_001,2,1,0
3,user_001,1,0,1


In [99]:
print(feedback_df.columns.tolist())
feedback_df.head()

['user_id', 'product_id', 'shown', 'clicked']


,user_id,product_id,shown,clicked
0,user_001,1,1,0
1,user_001,3,1,0
2,user_001,2,1,0
3,user_001,1,0,1


In [100]:
updated_stats = get_product_stats(products_df, feedback_df)
updated_stats[["product_name", "shown_count", "click_count", "click_rate"]].sort_values(
    by="click_rate",
    ascending=False
).head(10)


,product_name,shown_count,click_count,click_rate
0,Wireless Headphones,1,1,1.0
1,Bluetooth Speaker,1,0,0.0
2,Smart Watch,1,0,0.0
3,Running Shoes,0,0,0.0
4,Hoodie,0,0,0.0
5,Backpack,0,0,0.0
6,Coffee Maker,0,0,0.0
7,Air Fryer,0,0,0.0
8,Desk Lamp,0,0,0.0
9,Notebook Pack,0,0,0.0


In [101]:
user_preference = get_user_preference(user_id, feedback_df, products_df)
user_preference


'Electronics'

In [102]:
new_recommendations = choose_recommendations(
    user_id=user_id,
    products=products_df,
    feedback=feedback_df,
    category="All",
    top_n=3
)
new_recommendations


[{'product_id': 1,
  'product_name': 'Wireless Headphones',
  'category': 'Electronics',
  'brand': 'SonicX',
  'price': 79.99,
  'rating': 4.4,
  'description': 'Noise cancelling wireless headphones with long battery life',
  'shown_count': 1,
  'click_count': 1,
  'click_rate': 1.0},
 {'product_id': 3,
  'product_name': 'Smart Watch',
  'category': 'Electronics',
  'brand': 'TimeSync',
  'price': 129.99,
  'rating': 4.3,
  'description': 'Fitness tracking smart watch with heart rate monitor',
  'shown_count': 1,
  'click_count': 0,
  'click_rate': 0.0},
 {'product_id': 2,
  'product_name': 'Bluetooth Speaker',
  'category': 'Electronics',
  'brand': 'SoundBeat',
  'price': 49.99,
  'rating': 4.2,
  'description': 'Portable speaker with deep bass and waterproof design',
  'shown_count': 1,
  'click_count': 0,
  'click_rate': 0.0}]

In [103]:

"""The system recommends products from a catalog.
- Every recommendation is logged as an impression.
- User clicks are logged as rewards.
- The recommender uses epsilon-greedy bandit logic:
  - most of the time it recommends products with the best click performance
  - sometimes it explores random products
- Over time, recommendations improve based on user behavior"""

'The system recommends products from a catalog.\n- Every recommendation is logged as an impression.\n- User clicks are logged as rewards.\n- The recommender uses epsilon-greedy bandit logic:\n  - most of the time it recommends products with the best click performance\n  - sometimes it explores random products\n- Over time, recommendations improve based on user behavior'